# 01. Load and label

This notebook does two things. It takes a first look at the three data tables so you know what
you are working with, and it builds a **label table** that says, for every provider, whether they
were later excluded from Medicare for fraud-related reasons. Every model in this project is
trained on or scored against that label table.

If you are opening this project for the first time, read the next section before running anything.

## Where the data comes from

This notebook does not read any CSV directly. The raw files were parsed once by
`scripts/build_database.py` into `database/medicare_fraud.duckdb`, and every notebook works
from those tables. Parsing the 3.2 GB service file once, rather than on every notebook open,
is the whole reason for the database step.

| DuckDB table | raw CSV in `data_raw/` | grain | rows |
|---|---|---|---|
| `provider` | `MUP_PHY_R26_P05_V10_D24_Prov.csv` | one row per provider, 2024 totals | ~1.30M |
| `provider_service` | `PHY_R26_P05_V10_D24_Prov_Svc.csv` | one row per provider x procedure code x place of service, 2024 | ~9.78M |
| `leie` | `UPDATED.csv` | one row per exclusion | ~84K |

### Acronyms you will see throughout

| acronym | meaning |
|---|---|
| CMS | Centers for Medicare & Medicaid Services, the federal agency that runs Medicare |
| NPI | National Provider Identifier, the 10-digit ID every clinician and healthcare organization has. Our primary key |
| HCPCS | Healthcare Common Procedure Coding System, the codes providers put on bills to say what service they performed (for example 99213 is a mid-level office visit) |
| OIG | Office of Inspector General, the investigative arm of the Department of Health and Human Services (HHS) |
| LEIE | List of Excluded Individuals and Entities, the OIG's published list of people barred from federal health programs |
| Part B | The part of Medicare that covers outpatient care: office visits, procedures, labs, imaging, clinic-administered drugs |
| E&M | Evaluation and Management, the family of HCPCS codes for office and hospital visits |
| HCC risk score | Hierarchical Condition Category score, CMS's single number for how sick a patient is. 1.0 is the national average |
| FWA | Fraud, waste, and abuse, the umbrella term for improper Medicare payments |

### The two CMS files: what the providers did

Both come from the CMS **Medicare Physician & Other Practitioners** public use files for
calendar year 2024. CMS builds them from every Part B fee-for-service claim paid that year,
then aggregates so that no individual patient or claim is visible. Hospital stays (Part A),
prescriptions filled at a pharmacy (Part D), and Medicare Advantage are not in these files.

**`MUP_PHY_..._Prov.csv`, the "by Provider" file.** One row for each of the 1.3M clinicians
and organizations that billed Part B in 2024, identified by NPI. Each row totals everything
they did that year: how many patients, how many services, what they charged, what Medicare
paid, and a picture of their patient panel (age mix, share also on Medicaid, average risk
score, chronic condition rates). This file answers "how much did this provider do, and who
were their patients?"

Role in the project: the backbone table. It defines the population we score, supplies the
provider's specialty for peer comparison, and gives the volume, dollar, and patient-panel
features.

The columns in this file, grouped:

| group | columns | what they hold |
|---|---|---|
| identity | `Rndrng_NPI`, `Rndrng_Prvdr_Last_Org_Name`, `_First_Name`, `_MI`, `_Crdntls`, `_Ent_Cd` | NPI, name, credentials, I = individual or O = organization |
| location | `Rndrng_Prvdr_St1`, `_St2`, `_City`, `_State_Abrvtn`, `_State_FIPS`, `_Zip5`, `_RUCA`, `_RUCA_Desc`, `_Cntry` | practice address, state code, rural-urban code |
| specialty | `Rndrng_Prvdr_Type`, `Rndrng_Prvdr_Mdcr_Prtcptg_Ind` | billing specialty (our peer-group key), accepts Medicare fee schedule Y/N |
| yearly totals | `Tot_HCPCS_Cds`, `Tot_Benes`, `Tot_Srvcs`, `Tot_Sbmtd_Chrg`, `Tot_Mdcr_Alowd_Amt`, `Tot_Mdcr_Pymt_Amt`, `Tot_Mdcr_Stdzd_Amt` | distinct codes, patients, services, charged, allowed, paid, geography-adjusted paid |
| drug totals | `Drug_Sprsn_Ind` plus the same seven measures prefixed `Drug_` | the totals counting only drug codes; suppression flag when under 11 patients |
| medical totals | `Med_Sprsn_Ind` plus the same seven prefixed `Med_` | the totals counting only non-drug services |
| patient age | `Bene_Avg_Age`, `Bene_Age_LT_65_Cnt`, `_65_74_Cnt`, `_75_84_Cnt`, `_GT_84_Cnt` | mean age and count per age band |
| patient sex and race | `Bene_Feml_Cnt`, `Bene_Male_Cnt`, `Bene_Race_Wht_Cnt`, `_Black_Cnt`, `_API_Cnt`, `_Hspnc_Cnt`, `_NatInd_Cnt`, `_Othr_Cnt` | counts as recorded in Medicare enrollment |
| Medicaid overlap | `Bene_Dual_Cnt`, `Bene_Ndual_Cnt` | patients also on Medicaid (dual eligible) and not |
| behavioral health | `Bene_CC_BH_*_Pct` (11 columns) | percent of panel with ADHD, alcohol/drug, tobacco, dementia, anxiety, bipolar, mood, depression, personality, PTSD, schizophrenia |
| physical health | `Bene_CC_PH_*_Pct` (15 columns) | percent with asthma, afib, cancer, CKD, COPD, diabetes, heart failure, hyperlipidemia, hypertension, ischemic heart, osteoporosis, Parkinson's, arthritis, stroke |
| risk | `Bene_Avg_Risk_Scre` | mean HCC risk score, 1.0 = national average |

81 columns in all. `Docs/data_dictionary.md` has every one on its own line.


**`PHY_..._Prov_Svc.csv`, the "by Provider and Service" file.** The same providers, but broken
out by what they billed. Each row is one provider, one HCPCS procedure code, and one place of
service (office or facility). A cardiologist who billed 40 different codes has 40 rows here,
more if some were in the office and some in a hospital. Each row says how many patients got that
code, how many times it was billed, on how many distinct patient-days, and the average charge and
payment. This file answers "what exactly did this provider bill, and how often?"

Role in the project: the billing-mix table. Fraud rarely shows up as raw volume. It shows up as
an unusual pattern of codes: too many high-level office visits, the same code billed repeatedly
on one patient in one day, one code making up most of the revenue. Those features can only be
built from this file. It is too large to load into pandas, so it is aggregated down to one row
per provider in SQL.

The columns in this file:

| group | columns | what they hold |
|---|---|---|
| provider (repeated) | the first 17 columns of the provider file, identity through participation flag | identical on every row for the same NPI; ignore here and join to `provider` instead |
| the code | `HCPCS_Cd`, `HCPCS_Desc`, `HCPCS_Drug_Ind` | procedure code, plain-English description, Y if it is a drug |
| where | `Place_Of_Srvc` | F = facility (hospital, surgery center), O = office. Same code pays differently in each |
| volume | `Tot_Benes`, `Tot_Srvcs`, `Tot_Bene_Day_Srvcs` | patients who got this code, times billed, distinct patient-days billed |
| dollars | `Avg_Sbmtd_Chrg`, `Avg_Mdcr_Alowd_Amt`, `Avg_Mdcr_Pymt_Amt`, `Avg_Mdcr_Stdzd_Amt` | per-service charged, allowed, paid, geography-adjusted paid |

28 columns in all. The gap between `Tot_Srvcs` and `Tot_Bene_Day_Srvcs` is the same-day repeat
signal used in the features notebook.


Neither CMS file says anything about fraud. They are pure billing records for everyone,
honest or not.

### The OIG file: which providers were caught

**`UPDATED.csv`, the LEIE.** Published monthly by the OIG. Every row is a person or business
that has been barred from billing federal health programs, with the statutory reason
(`EXCLTYPE`), the effective date (`EXCLDATE`), and, where OIG has it, the NPI. Reasons range
from a felony conviction for Medicare fraud to a lapsed state license. Most rows carry no NPI
at all because they predate the NPI system or are businesses.

Role in the project: the label source, and nothing else. It contributes no features. It is
joined to the provider table on NPI so we can mark which 2024 billers were later excluded for
fraud-related reasons. That is the outcome every model in this project is scored against. It is
a *weak* label: some fraud is never caught, and being excluded is not the same as being
convicted for what showed up in the 2024 billing. But it is the only public, provider-level
fraud outcome that exists, and it is the one that program-integrity teams actually use.

The columns in this file:

| group | columns | what they hold |
|---|---|---|
| who | `LASTNAME`, `FIRSTNAME`, `MIDNAME`, `BUSNAME` | person's name, or business name for an entity |
| what kind | `GENERAL`, `SPECIALTY` | OIG's broad category and free-text specialty, not aligned to CMS provider types |
| identifiers | `UPIN`, `NPI`, `DOB` | obsolete pre-2007 ID (blank), the NPI or `0000000000` if unknown, date of birth |
| where | `ADDRESS`, `CITY`, `STATE`, `ZIP` | last known address |
| the exclusion | `EXCLTYPE`, `EXCLDATE` | statutory authority (e.g. 1128a1) and effective date as YYYYMMDD. These two decide the label |
| reversal | `REINDATE`, `WAIVERDATE`, `WVRSTATE` | reinstatement date (`00000000` if still excluded), state waiver date and state |

18 columns in all. Only `NPI`, `EXCLTYPE`, and `EXCLDATE` are used in this project.


### How they fit together

```
provider (who, how much)  ---+
                             |--> features, one row per NPI  ---> model score
provider_service (what)   ---+
                                                                     |
leie (who was caught)     ---> labels, one row per NPI  ------------> evaluation
```

The CMS files describe behavior. The LEIE describes outcomes. The model learns which behaviors
tend to precede the outcome.

### About DuckDB

DuckDB is a SQL database that runs inside Python with no server, like SQLite. Unlike SQLite it
stores data by column rather than by row, which makes it fast at the "group and sum a huge table"
work this project needs. The 9.8M row table aggregates in seconds. We do the heavy grouping in
SQL and only pull the small result into pandas.

## 0. Connect to the database

**What this cell does:** opens the DuckDB file and defines a helper function `q()` that runs a
SQL query and returns the result as a pandas DataFrame. Every later cell uses `q()`.

**Why:** so the rest of the notebook can be written as short SQL strings instead of repeating
connection boilerplate. The first query lists every table and its row count, which confirms
the build script worked before we do anything else.

In [1]:
import duckdb
import pandas as pd

DB_PATH = "C:/Users/palla/OneDrive/Documents/Coding Projects/Medicare Fraud ML/database/medicare_fraud.duckdb"
con = duckdb.connect(DB_PATH)
# DuckDB shows a live progress bar widget in notebooks. VS Code cannot render it and
# the kernel stalls while it tries, so turn it off. Purely cosmetic.
con.execute("SET enable_progress_bar = false")

def q(sql: str) -> pd.DataFrame:
    """Run a SQL query against the database and return the result as a DataFrame."""
    return con.sql(sql).df()

q("SELECT table_name, estimated_size AS approx_rows FROM duckdb_tables() ORDER BY 1")

,table_name,approx_rows
0,labels,1296739
1,leie,83842
2,provider,1296739
3,provider_features,1296739
4,provider_service,9781673


## 1. What a provider row looks like

**What this cell does:** pulls three rows from the `provider` table and flips them sideways
(`.T` is transpose) so the 81 column names run down the page with the three providers' values
beside them.

**Why:** the table is too wide to read normally. Before analyzing anything you should see what
one provider actually looks like: name, specialty, address, then yearly totals, then patient
demographics, then chronic-condition percentages. Every column is explained in
`Docs/data_dictionary.md`.

In [2]:
q("SELECT * FROM provider LIMIT 3").T

,0,1,2
Rndrng_NPI,1003000126,1003000134,1003000142
Rndrng_Prvdr_Last_Org_Name,Enkeshafi,Cibull,Khalil
Rndrng_Prvdr_First_Name,Ardalan,Thomas,Rashid
Rndrng_Prvdr_MI,NaN,L,NaN
Rndrng_Prvdr_Crdntls,M.D.,M.D.,M.D.
...,...,...,...
Bene_CC_PH_Osteoporosis_V2_Pct,18,15,17
Bene_CC_PH_Parkinson_V2_Pct,4,2,<NA>
Bene_CC_PH_Arthritis_V2_Pct,63,47,75
Bene_CC_PH_Stroke_TIA_V2_Pct,21,6,11


**What this cell does:** counts providers in each specialty (`Rndrng_Prvdr_Type`) and shows the
median number of services and median Medicare payment for each.

**Why:** two reasons. First, to see who is in the data. Nurse practitioners are the largest
group, not physicians. Second, to see how different specialties are from each other. Median
billing varies by an order of magnitude across specialties, which means "a lot of services" is
meaningless without knowing the specialty. That is why every feature later gets compared to
the provider's own peer group.

In [3]:
q("""
SELECT Rndrng_Prvdr_Type AS provider_type,
       count(*)                          AS n_providers,
       round(median(Tot_Srvcs))          AS median_services,
       round(median(Tot_Mdcr_Pymt_Amt))  AS median_payment
FROM provider
GROUP BY 1
ORDER BY n_providers DESC
LIMIT 15
""")

,provider_type,n_providers,median_services,median_payment
0,Nurse Practitioner,203212,253.0,13301.0
1,Physician Assistant,112514,235.0,12626.0
2,Internal Medicine,93627,623.0,44569.0
3,Family Practice,83639,568.0,31326.0
4,Physical Therapist in Private Practice,77901,1571.0,32377.0
5,Certified Registered Nurse Anesthetist (CRNA),53495,109.0,11828.0
6,Emergency Medicine,50101,330.0,31330.0
7,Anesthesiology,41082,194.0,22889.0
8,Diagnostic Radiology,32843,2949.0,82012.0
9,Chiropractic,32394,370.0,9376.0


## 2. The exclusion list, and why most of it cannot be used

The LEIE has ~84K rows but only ~8.8K carry a real NPI (National Provider Identifier, the
10-digit ID every clinician and healthcare organization has, and the key that links this file
to the CMS billing tables). The rest have `0000000000` in the NPI column, mostly businesses and exclusions that predate the NPI system. Only rows with a real NPI
can be joined to billing data, so the other 75K rows are useless to us.

`EXCLTYPE` is the section of the Social Security Act the exclusion was issued under. The ones
that matter here:

| code | meaning |
|---|---|
| 1128a1 | felony conviction, program-related crime (Medicare or Medicaid fraud) |
| 1128a2 | felony conviction, patient abuse or neglect |
| 1128a3 | felony conviction, healthcare fraud against any payer |
| 1128a4 | felony conviction, controlled substances |
| 1128b4 | license revoked or surrendered (often not fraud) |
| 1128b7 | fraud, kickbacks, other prohibited activities (civil finding) |

Section (a) exclusions are **mandatory** and follow a criminal conviction. Section (b) exclusions
are **permissive**, meaning OIG chose to exclude, and 1128b4 in particular is usually a state
licensing action rather than a fraud finding.

**What the next cell does:** keeps only LEIE rows with a real NPI and counts how many fall under
each exclusion type.

**Why:** to see which exclusion reasons are common among the rows we can actually use. This
decides how the label is defined in section 4.

In [4]:
q("""
SELECT EXCLTYPE, count(*) AS n
FROM leie
WHERE NPI <> '0000000000'
GROUP BY 1 ORDER BY n DESC
LIMIT 12
""")

,EXCLTYPE,n
0,1128a1,3379
1,1128b4,2636
2,1128a4,1022
3,1128a3,681
4,1128a2,402
5,1128b14,196
6,1128b7,194
7,1128b8,96
8,1128b5,85
9,1128b1,48


## 3. Joining the two

How many providers who billed Medicare in 2024 are on the exclusion list at all? And when were
they excluded relative to the billing year?

The timing question matters. A provider excluded in 2019 is barred from billing, so they will
not be in the 2024 file. The providers we *can* see are the ones excluded during or after 2024,
whose 2024 billing therefore happened while the conduct was still undetected. That is precisely
the population a fraud model is supposed to find.

**What the next cell does:** joins `provider` to `leie` on NPI and counts matches by exclusion
year and exclusion type. The NPI is stored as a number in the CMS table and as a text string in
the LEIE, so the CMS side is cast to text before matching.

**Why:** to confirm the join works, see how many providers match (it is small), and check the
timing story above against the data.

In [5]:
q("""
SELECT substr(l.EXCLDATE, 1, 4) AS exclusion_year,
       l.EXCLTYPE,
       count(DISTINCT p.Rndrng_NPI) AS n_providers
FROM provider p
JOIN leie l ON CAST(p.Rndrng_NPI AS VARCHAR) = l.NPI
GROUP BY 1, 2
ORDER BY 1, 3 DESC
""")

,exclusion_year,EXCLTYPE,n_providers
0,2015,1128a1,1
1,2024,1128a1,4
2,2024,1128a4,3
3,2024,1128b4,2
4,2024,1128b7,2
5,2024,1128b1,2
6,2024,1128b3,1
7,2025,1128b4,24
8,2025,1128a1,21
9,2025,1128a4,5


## 4. Defining the label

Decision: a provider is a **positive** (label = 1) if they have a section (a) mandatory
exclusion (1128a1, a2, a3, a4) or a 1128b7 fraud exclusion, dated 2024 or later.

A provider is a **negative** (label = 0) if they never appear in the LEIE at all.

Everyone else gets **no label** (NULL) and is dropped from modeling. That covers license
revocations (1128b4) and any exclusion dated before 2024. They are ambiguous, so we set them
aside rather than guess.

This gives a very small positive class against ~1.3M negatives. That is the real shape of the
problem in program integrity work. It is why accuracy is useless here (a model that says "nobody
is fraudulent" is 99.99% accurate) and why the evaluation notebook uses ranking metrics instead:
ROC AUC, PR AUC, and precision at k.

**What the next cell does:** this is the only cell in the notebook that writes to the database.
It creates a table called `labels` with one row per provider. Step by step:

1. The `WITH excl AS (...)` block collapses the LEIE to one row per NPI, keeping the earliest
   exclusion date and two true/false flags: does this NPI have any fraud-type exclusion, and
   does it have any license-type exclusion.
2. The main `SELECT` takes every provider and left-joins that summary, so providers with no
   exclusion get NULLs from the LEIE side.
3. The `CASE` expression turns that into the three-way label described above.

**Why write it to the database:** so every later notebook reads the same definition of "fraud."
If the label lived in a notebook variable, each notebook could drift to its own version.

In [6]:
con.execute("""
CREATE OR REPLACE TABLE labels AS
WITH excl AS (
    SELECT NPI,
           min(EXCLDATE) AS first_excl_date,
           bool_or(EXCLTYPE IN ('1128a1','1128a2','1128a3','1128a4','1128b7')) AS is_fraud_type,
           bool_or(EXCLTYPE = '1128b4') AS is_license_type
    FROM leie
    WHERE NPI <> '0000000000'
    GROUP BY NPI
)
SELECT p.Rndrng_NPI AS npi,
       CASE
         WHEN e.NPI IS NULL THEN 0                                   -- never excluded
         WHEN e.is_fraud_type AND e.first_excl_date >= '20240101' THEN 1
         ELSE NULL                                                   -- ambiguous: drop from modeling
       END AS label,
       e.first_excl_date,
       e.is_fraud_type,
       e.is_license_type
FROM provider p
LEFT JOIN excl e ON CAST(p.Rndrng_NPI AS VARCHAR) = e.NPI
""")

q("""
SELECT label, count(*) AS n
FROM labels
GROUP BY 1 ORDER BY 1 NULLS LAST
""")

,label,n
0,0,1296590
1,1,68
2,<NA>,81


## 5. Sanity check: do the positives look different at all?

**What this cell does:** joins the new `labels` table back to `provider` and compares the median
of a few raw totals between label 0 and label 1.

**Why:** before building any features or models, it is worth knowing whether the excluded
providers are distinguishable on the simplest possible measures. If they looked identical to
everyone else on volume and dollars, the features would have to come entirely from billing
*mix* in the `provider_service` table.

What it shows: excluded providers do not bill more in total. Their median services and payments
are slightly below everyone else's. But their median services **per beneficiary** is well above
(5.1 versus 2.9). Intensity per patient, not raw volume, is the first signal, and it is what
points the feature engineering in the next notebook toward ratios rather than totals.

In [7]:
q("""
SELECT lb.label,
       count(*)                                        AS n,
       round(median(p.Tot_Benes))                      AS med_benes,
       round(median(p.Tot_Srvcs))                      AS med_services,
       round(median(p.Tot_Srvcs / p.Tot_Benes), 1)     AS med_services_per_bene,
       round(median(p.Tot_Mdcr_Pymt_Amt))              AS med_payment,
       round(median(p.Tot_HCPCS_Cds))                  AS med_distinct_codes
FROM labels lb
JOIN provider p ON p.Rndrng_NPI = lb.npi
WHERE lb.label IS NOT NULL
GROUP BY 1 ORDER BY 1
""")

,label,n,med_benes,med_services,med_services_per_bene,med_payment,med_distinct_codes
0,0,1296590,130.0,433.0,2.9,26514.0,16.0
1,1,68,117.0,425.0,5.1,24417.0,15.0


## 6. Close the connection

**What this cell does:** releases the database file so other notebooks and scripts can open it.
DuckDB allows only one writer at a time, so leaving a connection open here would block the
features notebook from writing its table.

In [8]:
con.close()